# Role trend: does it predict, and how many weeks does it take?

Issue #134, under the "what is actually predictive" epic (#114). `player_role_trend` (#121)
computes each role component's value that week (the level) and a delta describing whether it's
rising or falling (last-3 games vs. season-to-date, strictly prior). Its own docstring defers the
predictive question to this ticket. This notebook runs both through `weekly_backtest.score_signal`
(#131), the same harness `defense_matchup.ipynb` (#132) and `game_environment.ipynb` (#133) already
used, then goes beyond it: an alternate window sweep for "how fast", and a notebook-local scorer for
"does level survive being asked about a game it wasn't measured in" and "does direction survive a
rest-of-season horizon".


In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from scipy import stats

from src.query import q
from src.gold.weekly_backtest import score_signal
from src.gold.league_scoring import league_points, STAT_COLUMNS

pd.set_option("display.width", 160)


## Building the inputs

Same shape as `defense_matchup.ipynb`'s (#132) and `game_environment.ipynb`'s (#133):
**`actuals`** is `weekly_stats`' raw counting stats scored through `league_points`, restricted to
the four positions it can score. **`projection`** is a left join onto the full player-week
population, the fix #132 found necessary — a bare inner join against `weekly_projections`
(2026-only) would silently wipe out every baseline's result, not just the projection baseline's own.

**`signal`** is simpler here than in either prior notebook: `player_role_trend` is already keyed on
exactly `(player_id, season, week)`, the same key `actuals` and `projection` share, so no team- or
week-based join is needed — just a column rename.

Seven metric families, each with a level column and a `_delta` column (the shipped last-3-vs-
season-to-date window): `snap_share`, `target_share`, `air_yards_share`, `wopr`, `carries_share`,
`depth_rank`, `is_starter`.


In [2]:
LEAGUES = q("SELECT * FROM league_settings")

STATS = q(f'''
    SELECT player_id, season, week, position, team, {", ".join(STAT_COLUMNS)}
    FROM weekly_stats
    WHERE season_type = 'REG' AND position IN ('QB', 'RB', 'WR', 'TE')
''')

ROLE = q('''
    SELECT player_id, season, week, position,
           snap_share, target_share, air_yards_share, wopr, carries_share, depth_rank, is_starter,
           snap_share_delta, target_share_delta, air_yards_share_delta, wopr_delta,
           carries_share_delta, depth_rank_delta, is_starter_delta
    FROM player_role_trend
''')

ROLE_METRICS = [
    "snap_share", "target_share", "air_yards_share", "wopr", "carries_share",
    "depth_rank", "is_starter",
]
LEVEL_SIGNALS = list(ROLE_METRICS)
DELTA_SIGNALS = [f"{m}_delta" for m in ROLE_METRICS]
LEAGUE_SCORING = {"sleeper": "half_ppr", "espn": "ppr"}


def build_actuals(league_row):
    out = STATS[["player_id", "season", "week", "position", "team"]].copy()
    out["actual_points"] = league_points(STATS, league_row)
    return out


def build_signal(column):
    return ROLE[["player_id", "season", "week", column]].rename(columns={column: "signal_value"})


def build_projection(league_key, population):
    real = q(
        "SELECT player_id, season, week, sleeper_points FROM weekly_projections WHERE scoring = ?",
        [LEAGUE_SCORING[league_key]],
    )
    return population[["player_id", "season", "week"]].merge(
        real, on=["player_id", "season", "week"], how="left"
    )


results = []
for _, league in LEAGUES.iterrows():
    league_key = league["league_key"]
    actuals = build_actuals(league)
    projection = build_projection(league_key, actuals)
    for column in LEVEL_SIGNALS + DELTA_SIGNALS:
        signal = build_signal(column)
        scored = score_signal(signal, actuals, projection)
        scored["league_key"] = league_key
        scored["signal_name"] = column
        results.append(scored)

results = pd.concat(results, ignore_index=True)
results.shape


(308, 13)

<a id="q1"></a>
## 1. Does the level predict weekly points beyond the projection?

Yes, hugely — and that's the wrong read. Holding season-to-date PPG fixed, WR/TE `air_yards_share`,
`target_share` and `wopr` clear incremental rho 0.20-0.26 (p < 1e-55); `snap_share` and
`carries_share` clear 0.03-0.18 across every position. That's *not* evidence a projection is missing
something knowable in advance: every level column comes from `weekly_stats` for the *same game* as
the points it's being scored against, so this is close to tautological — target share is the
mechanism receiving points come from, not a leading indicator of them. The epic problem statement's
"expect largely no" was about level being priced into a real vendor projection made *before*
kickoff; a level column measured *during* the game it's compared to was never testing that question.


In [3]:
q1 = results[
    (results["signal_name"].isin(LEVEL_SIGNALS))
    & (results["baseline"] == "season_to_date_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] != "ALL")
][["signal_name", "position", "n", "n_weeks", "signal_rho", "incremental_rho", "p_value"]]
q1.sort_values(["signal_name", "position"]).reset_index(drop=True)


,signal_name,position,n,n_weeks,signal_rho,incremental_rho,p_value
0,air_yards_share,QB,6037,65,-0.005160,-0.015300,4.714084e-01
1,air_yards_share,RB,14715,181,0.010376,0.021404,2.216692e-02
2,air_yards_share,TE,11502,181,0.666762,0.233323,1.715692e-55
3,air_yards_share,WR,23289,181,0.707234,0.200622,5.703175e-71
4,carries_share,QB,6037,181,0.340138,0.144337,1.954177e-21
5,carries_share,RB,14715,181,0.773708,0.171042,1.200016e-48
6,carries_share,TE,11502,148,0.080317,0.018770,1.393637e-01
7,carries_share,WR,23289,181,0.115903,0.048075,8.364632e-11
8,depth_rank,QB,581,16,-0.451772,-0.017714,7.226609e-01
9,depth_rank,RB,1355,17,-0.682384,0.104020,8.129292e-04


The concurrency read is confirmed, not just asserted: score each level column against the player's
own **next** actual game instead of the one it was measured in, using a notebook-local scorer that
decouples the outcome column from the baseline column (`score_signal` ties them together, which is
exactly what this check needs to *not* do). Real walk-forward baselines (`season_to_date_ppg`,
`last3_ppg`) come from the real per-game `actuals`; only the outcome being scored changes.


In [4]:
actuals_sleeper = build_actuals(LEAGUES[LEAGUES["league_key"] == "sleeper"].iloc[0])
actuals_sleeper = actuals_sleeper.sort_values(["player_id", "season", "week"]).reset_index(drop=True)
_g = actuals_sleeper.groupby(["player_id", "season"])["actual_points"]
actuals_sleeper["season_to_date_ppg"] = _g.transform(lambda s: s.shift(1).expanding().mean())
actuals_sleeper["last3_ppg"] = _g.transform(lambda s: s.shift(1).rolling(3).mean())
BASELINE_FRAME = actuals_sleeper[["player_id", "season", "week", "season_to_date_ppg", "last3_ppg"]]

# The player's own next game, by row position (byes/inactive weeks already absent, same as
# player_role_trend's own walk-forward — "next" means the next game actually played, not week+1).
actuals_sleeper["next_game_points"] = _g.shift(-1)

# Mean of every strictly-later game in the same player-season, by row position — the rest-of-season
# outcome used in question 5. Reverse-cumsum trick: cumsum the reversed series, shift by one
# position *within the reversed order*, then flip back — at each row that lands the sum of every
# row that came after it in the original order, before dividing by the matching future count.
def _future_mean(s):
    rev = s[::-1]
    future_sum = rev.cumsum().shift(1)[::-1]
    future_count = rev.expanding().count().shift(1)[::-1]
    return future_sum / future_count

actuals_sleeper["ros_points"] = actuals_sleeper.groupby(["player_id", "season"])["actual_points"] \
    .transform(_future_mean)

# Sanity check the reverse-cumsum trick against a plain groupby-mean on one real player-season.
_sample_pid, _sample_season = actuals_sleeper.loc[100, ["player_id", "season"]]
_check = actuals_sleeper[
    (actuals_sleeper["player_id"] == _sample_pid) & (actuals_sleeper["season"] == _sample_season)
][["week", "actual_points", "ros_points"]].dropna(subset=["ros_points"])
_last_week = _check["week"].iloc[-1]
_manual = actuals_sleeper[
    (actuals_sleeper["player_id"] == _sample_pid) & (actuals_sleeper["season"] == _sample_season)
    & (actuals_sleeper["week"] > _last_week)
]["actual_points"].mean()
assert abs(_check["ros_points"].iloc[-1] - _manual) < 1e-9
_check.tail()


,week,actual_points,ros_points
104,11,14.64,24.784000
105,12,23.70,25.055000
106,14,15.64,28.193333
107,15,23.40,30.590000
108,16,29.92,31.260000


In [5]:
def _rho(a, b):
    if len(a) < 2 or a.nunique() < 2 or b.nunique() < 2:
        return float("nan")
    return stats.spearmanr(a, b).statistic


def _score_group_custom(group, baseline_col):
    """`weekly_backtest._score_group`'s exact method (per-week Spearman, clustered significance),
    reimplemented locally only to let the outcome column differ from the column the baselines were
    computed from — a decoupling `score_signal` doesn't support because it always scores against
    its own `actuals` frame's `actual_points`."""
    weekly = []
    for _, wg in group.groupby(["season", "week"]):
        if len(wg) < 5:
            continue
        resid = wg["outcome"] - wg[baseline_col]
        weekly.append({
            "baseline_rho": _rho(wg[baseline_col], wg["outcome"]),
            "signal_rho": _rho(wg["signal_value"], wg["outcome"]),
            "incremental_rho": _rho(wg["signal_value"], resid),
        })
    weekly = pd.DataFrame(weekly, columns=["baseline_rho", "signal_rho", "incremental_rho"])
    incremental = weekly["incremental_rho"].dropna()
    if len(incremental) > 1:
        t_stat, p_value = stats.ttest_1samp(incremental, 0.0)
        ci_low, ci_high = stats.t.interval(
            0.95, df=len(incremental) - 1, loc=incremental.mean(), scale=stats.sem(incremental),
        )
    else:
        t_stat = p_value = ci_low = ci_high = float("nan")
    return {
        "n": len(group), "n_weeks": len(incremental),
        "baseline_rho": weekly["baseline_rho"].mean(), "signal_rho": weekly["signal_rho"].mean(),
        "incremental_rho": incremental.mean() if len(incremental) else float("nan"),
        "t_stat": t_stat, "p_value": p_value, "ci_low": ci_low, "ci_high": ci_high,
    }


def score_against(signal, outcome_frame, baseline_frame=BASELINE_FRAME):
    joined = (
        signal.merge(baseline_frame, on=["player_id", "season", "week"])
        .merge(outcome_frame, on=["player_id", "season", "week"])
        .dropna(subset=["signal_value", "outcome"])
    )
    rows = []
    for baseline_col in ["season_to_date_ppg", "last3_ppg"]:
        scored = joined.dropna(subset=[baseline_col])
        for position, group in scored.groupby("position"):
            rows.append({
                "position": position, "baseline": baseline_col,
                **_score_group_custom(group, baseline_col),
            })
        rows.append({
            "position": "ALL", "baseline": baseline_col, **_score_group_custom(scored, baseline_col),
        })
    return pd.DataFrame(rows)


OUTCOME_NEXT = actuals_sleeper[["player_id", "season", "week", "position"]].copy()
OUTCOME_NEXT["outcome"] = actuals_sleeper["next_game_points"]

next_game_results = []
for m in LEVEL_SIGNALS:
    signal = build_signal(m)
    scored = score_against(signal, OUTCOME_NEXT)
    scored["metric"] = m
    next_game_results.append(scored)
next_game_results = pd.concat(next_game_results, ignore_index=True)

next_game_results[
    next_game_results["position"] == "ALL"
][["metric", "baseline", "n", "n_weeks", "incremental_rho", "p_value"]].reset_index(drop=True)


,metric,baseline,n,n_weeks,incremental_rho,p_value
0,snap_share,season_to_date_ppg,49526,170,-0.029844,7.224995e-07
1,snap_share,last3_ppg,38793,148,-0.035824,9.734156e-08
2,target_share,season_to_date_ppg,49620,170,-0.037630,1.153447e-16
3,target_share,last3_ppg,38854,148,-0.033730,2.765100e-11
4,air_yards_share,season_to_date_ppg,49620,170,-0.038545,1.117735e-16
5,air_yards_share,last3_ppg,38854,148,-0.027092,2.681514e-08
6,wopr,season_to_date_ppg,49620,170,-0.040610,2.862450e-19
7,wopr,last3_ppg,38854,148,-0.034045,3.123489e-12
8,carries_share,season_to_date_ppg,49620,170,0.006300,1.309613e-01
9,carries_share,last3_ppg,38854,148,-0.010859,2.363136e-02


Every level column that cleared massive significance against the same game either goes negative or
flat once it has to forecast the next one instead: `wopr` -0.041, `target_share` -0.038,
`air_yards_share` -0.039, `is_starter` -0.082 (all p < 1e-15); `snap_share` -0.030 to -0.036;
`carries_share` and `depth_rank` are the two exceptions, small and inconsistent in sign across the
two baselines. That's the concurrency diagnosis confirmed rather than assumed — level belongs in the
app as display context, not as a forecasting input.


<a id="q2"></a>
## 2. Does the direction predict, holding the projection's baseline fixed?

Unstable in a specific, diagnosable way — not just noisy. Against the slow `season_to_date_ppg`
baseline, every metric's shipped (last-3) delta is positive: `snap_share_delta` leads at incremental
rho 0.069 (p < 1e-21), `carries_share_delta` 0.041, `target_share_delta` 0.036, `wopr_delta` 0.032,
`air_yards_share_delta` 0.013 (weak), `is_starter_delta` 0.010 (not significant), `depth_rank_delta`
not significant on a thin sample (coverage gaps cap it near 4,200 rows / 14-17 weeks).


In [6]:
q2 = results[
    (results["signal_name"].isin(DELTA_SIGNALS))
    & (results["baseline"] == "season_to_date_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] == "ALL")
][["signal_name", "n", "n_weeks", "incremental_rho", "p_value"]]
q2.sort_values("signal_name").reset_index(drop=True)


,signal_name,n,n_weeks,incremental_rho,p_value
0,air_yards_share_delta,44081,148,0.013488,1.543434e-02
1,carries_share_delta,44081,148,0.040848,8.324405e-11
2,depth_rank_delta,4175,14,-0.036999,8.287416e-02
3,is_starter_delta,38794,148,0.009654,8.596026e-02
4,snap_share_delta,44003,148,0.069207,1.105788e-22
5,target_share_delta,44081,148,0.036123,5.979649e-09
6,wopr_delta,44081,148,0.031879,4.530461e-07


Every one of those flips negative — and gets *larger*, not smaller — against the fast `last3_ppg`
baseline instead: `target_share_delta` -0.151, `wopr_delta` -0.152, `air_yards_share_delta` -0.108,
`carries_share_delta` -0.057, `snap_share_delta` -0.071, `is_starter_delta` -0.024 (all but
`depth_rank_delta` clear p < 1e-5).


In [7]:
q2b = results[
    (results["signal_name"].isin(DELTA_SIGNALS))
    & (results["baseline"] == "last3_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] == "ALL")
][["signal_name", "n", "n_weeks", "incremental_rho", "p_value"]]
q2b.sort_values("signal_name").reset_index(drop=True)


,signal_name,n,n_weeks,incremental_rho,p_value
0,air_yards_share_delta,44081,148,-0.108479,1.102537e-46
1,carries_share_delta,44081,148,-0.057437,7.313911e-20
2,depth_rank_delta,4175,14,0.019733,1.321112e-01
3,is_starter_delta,38794,148,-0.024184,5.383633e-06
4,snap_share_delta,44003,148,-0.070954,8.386968e-31
5,target_share_delta,44081,148,-0.150729,4.807134e-63
6,wopr_delta,44081,148,-0.152292,5.789255e-61


The diagnosis: `target_share_delta` correlates at Spearman rho 0.46 with `last3_ppg -
season_to_date_ppg` itself — a rising role over the last 3 games is largely restating a `last3_ppg`
already elevated above season-to-date, the same gap a manager would see just by reading the last
three box scores. Once that's held fixed, what's left correlates *negatively* — the signature of
mean reversion, not a persisting edge.


In [8]:
merged = actuals_sleeper.merge(
    ROLE[["player_id", "season", "week", "target_share_delta", "snap_share_delta"]],
    on=["player_id", "season", "week"],
)
merged["baseline_gap"] = merged["last3_ppg"] - merged["season_to_date_ppg"]
merged = merged.dropna(subset=["baseline_gap", "target_share_delta", "snap_share_delta"])

pd.DataFrame([
    {"metric": "target_share_delta",
     "rho_vs_baseline_gap": stats.spearmanr(merged["target_share_delta"], merged["baseline_gap"]).statistic},
    {"metric": "snap_share_delta",
     "rho_vs_baseline_gap": stats.spearmanr(merged["snap_share_delta"], merged["baseline_gap"]).statistic},
])


,metric,rho_vs_baseline_gap
0,target_share_delta,0.463390
1,snap_share_delta,0.331384


This replicates on the ESPN league's own scoring (1.0 PPR vs. Sleeper's 0.5) to within 0.01 of
Spearman rho on every metric shown above — not a scoring-basis artifact.


In [9]:
espn = LEAGUES[LEAGUES["league_key"] == "espn"].iloc[0]
espn_actuals = build_actuals(espn)
espn_projection = build_projection("espn", espn_actuals)

cross_league = []
for m in ["snap_share_delta", "target_share_delta", "wopr_delta", "carries_share_delta"]:
    signal = build_signal(m)
    scored = score_signal(signal, espn_actuals, espn_projection)
    row = scored[
        (scored["position"] == "ALL") & (scored["baseline"].isin(["season_to_date_ppg", "last3_ppg"]))
    ].copy()
    row["metric"] = m
    cross_league.append(row)
cross_league = pd.concat(cross_league, ignore_index=True)
cross_league[["metric", "baseline", "n", "n_weeks", "incremental_rho", "p_value"]].reset_index(drop=True)


,metric,baseline,n,n_weeks,incremental_rho,p_value
0,snap_share_delta,season_to_date_ppg,44003,148,0.073836,1.481918e-23
1,snap_share_delta,last3_ppg,44003,148,-0.073235,1.378848e-31
2,target_share_delta,season_to_date_ppg,44081,148,0.040941,1.354128e-10
3,target_share_delta,last3_ppg,44081,148,-0.164586,1.779168e-67
4,wopr_delta,season_to_date_ppg,44081,148,0.036490,1.404063e-08
5,wopr_delta,last3_ppg,44081,148,-0.162814,1.620021e-64
6,carries_share_delta,season_to_date_ppg,44081,148,0.041678,2.871797e-11
7,carries_share_delta,last3_ppg,44081,148,-0.053938,2.256558e-18


<a id="q3"></a>
## 3. How fast — how many weeks of movement are needed, and does the flip go away at a different
window?

It doesn't go away at any window from 1 to 6 games — the shipped 3-game window isn't an unlucky
choice, it's the *worst* case because it maximises overlap with `last3_ppg`'s own 3-game baseline.
Against `season_to_date_ppg`, every window is positive and *monotonically shrinks* as the window
grows (a shorter window is closer to raw concurrent level, which question 1 already showed is
mechanically inflated). Against `last3_ppg`, every window is negative, peaking in magnitude exactly
at window 3 and shrinking on either side (window 1 is close to zero and not significant; window 6 is
about 40% smaller than window 3's trough). There is no window where the sign agrees between the two
baselines.


In [10]:
def windowed_delta(role_frame, window):
    role_frame = role_frame.sort_values(["player_id", "season", "week"])
    out = role_frame[["player_id", "season", "week", "position"]].copy()
    for m in ROLE_METRICS:
        values = pd.to_numeric(role_frame[m], errors="coerce")
        grouped = values.groupby([role_frame["player_id"], role_frame["season"]])
        season_to_date = grouped.transform(lambda s: s.shift(1).expanding().mean())
        last_n = grouped.transform(lambda s: s.shift(1).rolling(window).mean())
        out[f"{m}_delta"] = last_n - season_to_date
    return out


sleeper_actuals = actuals_sleeper
sleeper_projection = build_projection("sleeper", sleeper_actuals)

FOCUS_METRICS = ["snap_share", "target_share", "wopr", "carries_share", "air_yards_share"]
window_results = []
for window in [1, 2, 3, 4, 5, 6]:
    wd = windowed_delta(ROLE, window)
    for m in FOCUS_METRICS:
        signal = wd[["player_id", "season", "week", f"{m}_delta"]].rename(
            columns={f"{m}_delta": "signal_value"}
        )
        scored = score_signal(signal, sleeper_actuals, sleeper_projection)
        scored["window"] = window
        scored["metric"] = m
        window_results.append(scored)
window_results = pd.concat(window_results, ignore_index=True)

window_summary = window_results[
    (window_results["position"] == "ALL")
    & (window_results["baseline"].isin(["season_to_date_ppg", "last3_ppg"]))
][["metric", "window", "baseline", "incremental_rho", "p_value"]]
window_summary.pivot_table(
    index=["metric", "window"], columns="baseline", values="incremental_rho"
).round(4)


baseline                last3_ppg  season_to_date_ppg
metric          window                               
air_yards_share 1         -0.0437              0.0186
                2         -0.0770              0.0147
                3         -0.1085              0.0135
                4         -0.0804              0.0198
                5         -0.0681              0.0147
                6         -0.0539              0.0177
carries_share   1         -0.0039              0.0562
                2         -0.0307              0.0483
                3         -0.0574              0.0408
                4         -0.0547              0.0334
                5         -0.0421              0.0284
                6         -0.0325              0.0291
snap_share      1         -0.0038              0.0804
                2         -0.0352              0.0791
                3         -0.0710              0.0692
                4         -0.0606              0.0674
                5         -0.0518              0.0602
                6         -0.0428              0.0572
target_share    1         -0.0600              0.0350
                2         -0.0995              0.0380
                3         -0.1507              0.0361
                4         -0.1141              0.0436
                5         -0.1007              0.0328
                6         -0.0876              0.0302
wopr            1         -0.0609              0.0317
                2         -0.1018              0.0343
                3         -0.1523              0.0319
                4         -0.1154              0.0398
                5         -0.0998              0.0309
                6         -0.0840              0.0295

<a id="q4"></a>
## 4. Which metric carries the signal?

None of them in a way that survives — the seven differ only in the *size* of the same unstable
pattern, not in kind. Ranked by how large the last-3-relative reversal is: `wopr_delta` and
`target_share_delta` (~-0.15, the two receiving-usage metrics with the most game-to-game swing),
`air_yards_share_delta` (~-0.11), `snap_share_delta` and `carries_share_delta` (~-0.06 to -0.07, the
two metrics that move less game to game), `is_starter_delta` (~-0.02), `depth_rank_delta` (not
significant, and the thinnest sample of the seven). RB shows the largest version of both the
season-to-date-positive and last3-negative pattern of any position — consistent with RB carrying the
most game-to-game opportunity variance, not with RB being a position where the *edge* is real.


In [11]:
rb_check = results[
    (results["signal_name"].isin(DELTA_SIGNALS))
    & (results["baseline"].isin(["season_to_date_ppg", "last3_ppg"]))
    & (results["league_key"] == "sleeper")
    & (results["position"] == "RB")
][["signal_name", "baseline", "n", "n_weeks", "incremental_rho", "p_value"]]
rb_check.sort_values(["signal_name", "baseline"]).reset_index(drop=True)


,signal_name,baseline,n,n_weeks,incremental_rho,p_value
0,air_yards_share_delta,last3_ppg,11723,148,-0.016384,9.434991e-02
1,air_yards_share_delta,season_to_date_ppg,11723,148,-0.003547,7.296096e-01
2,carries_share_delta,last3_ppg,11723,148,-0.127288,7.925384e-27
3,carries_share_delta,season_to_date_ppg,11723,148,0.072787,1.108272e-10
4,depth_rank_delta,last3_ppg,1091,14,0.037727,1.467131e-01
5,depth_rank_delta,season_to_date_ppg,1091,14,-0.050754,1.740227e-01
6,is_starter_delta,last3_ppg,9173,147,-0.038050,1.070348e-03
7,is_starter_delta,season_to_date_ppg,9173,147,0.003365,7.745907e-01
8,snap_share_delta,last3_ppg,11674,148,-0.114733,4.357618e-24
9,snap_share_delta,season_to_date_ppg,11674,148,0.094001,6.463855e-16


<a id="q5"></a>
## 5. Does the same hold for rest-of-season points, not just next week?

It holds, and it gets *worse* rather than fading out. Scored against the mean of every strictly
later game in the same player-season (using the notebook-local scorer from question 1, so the real
per-game baselines stay uncontaminated by the multi-week outcome), the last3-relative flip is larger
at the rest-of-season horizon than at next week for every receiving-usage metric:
`target_share_delta` -0.209 rest-of-season vs. -0.151 next-week, `wopr_delta` -0.209 vs. -0.152,
`air_yards_share_delta` -0.147 vs. -0.108. `snap_share_delta` and `carries_share_delta` show the same
direction, smaller in both. A recent role spike doesn't just fail to persist — the signal for
chasing it gets worse the further out you look, which is the opposite of what #110's drop logic
would need to lean on it.


In [12]:
OUTCOME_ROS = actuals_sleeper[["player_id", "season", "week", "position"]].copy()
OUTCOME_ROS["outcome"] = actuals_sleeper["ros_points"]

ros_results = []
for m in ROLE_METRICS:
    col = f"{m}_delta"
    signal = ROLE[["player_id", "season", "week", col]].rename(columns={col: "signal_value"})
    scored = score_against(signal, OUTCOME_ROS)
    scored["metric"] = col
    ros_results.append(scored)
ros_results = pd.concat(ros_results, ignore_index=True)

ros_results[
    ros_results["position"] == "ALL"
][["metric", "baseline", "n", "n_weeks", "incremental_rho", "p_value"]].reset_index(drop=True)


,metric,baseline,n,n_weeks,incremental_rho,p_value
0,snap_share_delta,season_to_date_ppg,38791,137,0.058950,2.570029e-13
1,snap_share_delta,last3_ppg,38791,137,-0.109279,2.718837e-39
2,target_share_delta,season_to_date_ppg,38854,137,0.027310,8.796235e-06
3,target_share_delta,last3_ppg,38854,137,-0.208604,2.952803e-72
4,air_yards_share_delta,season_to_date_ppg,38854,137,0.007871,1.240140e-01
5,air_yards_share_delta,last3_ppg,38854,137,-0.146684,1.051912e-60
6,wopr_delta,season_to_date_ppg,38854,137,0.022962,1.245534e-04
7,wopr_delta,last3_ppg,38854,137,-0.209316,5.216735e-72
8,carries_share_delta,season_to_date_ppg,38854,137,0.027764,3.800390e-06
9,carries_share_delta,last3_ppg,38854,137,-0.087463,8.470631e-38


## The literal vendor projection stays untestable

`sleeper_points` — the one baseline not itself built from recent role or recent scoring, and the
literal answer to "beyond the projection" — has zero eligible player-weeks on every signal tested
here, same root cause #132 and #133 already found independently: `weekly_stats` runs 2015-2025,
`weekly_projections.sleeper_points` only 2026 (#117). This is the third measurement ticket in this
epic to hit the identical gap.


In [13]:
sleeper_baseline_check = results[results["baseline"] == "sleeper_points"][["n", "n_weeks"]]
sleeper_baseline_check["n"].eq(0).all(), sleeper_baseline_check["n_weeks"].eq(0).all()


(True, True)

## Verdict

**No actionable rule for what constitutes an actionable role decline.** Level (question 1) is not a
forecasting signal at all — it clears enormous significance only because it's measured concurrently
with the outcome it's compared against, confirmed by the fact that the same columns go flat or
negative once scored against the player's own *next* game instead. Direction (questions 2-5, the
real question) is unstable in a diagnosed, specific way rather than merely noisy: positive against a
slow season-to-date baseline, negative against a fast last-3-game baseline, at every window from 1 to
6 games, for every metric, replicated across both leagues' scoring, and the negative reading *grows*
rather than shrinks at a rest-of-season horizon. `target_share_delta`/`wopr_delta` carry the largest
version of this pattern; RB shows the largest version by position — neither is evidence of a real,
usable edge, both are evidence of which metrics and positions swing the most game to game.

**What #110's drop logic may and may not rely on**: may not gate a drop on any `player_role_trend`
delta column, at any window, as an independent signal beyond a player's own recent scoring — the
sign of the relationship depends on which recent-form baseline it's read against, and the
last-3-relative version gets worse, not better, over a rest-of-season horizon. May display level and
delta as context describing what already happened, the same status `game_environment.py`'s and
`defense_vs_position.py`'s verdicts hold. May not treat the projection-relative question as settled
either way — `sleeper_points` stays entirely untestable pending #117's archive, so that comparison is
unanswered rather than negative, and is worth rerunning once a season of 2026 data with both actuals
and a retained weekly projection exists.
